`08_data_tracking.ipynb`: Visualización de datos y transformación a formato tracking de YOLO.

# Importaciones

In [1]:
import os
from pathlib import Path
import cv2 

from dataset.scanner import scan_raw_dataset
from dataset.analyzer import parse_annotations
from dataset.yolo_converter import process_video_to_yolo
from utils.file_utils import load_gt
from dataset.splitter import (
    get_processed_videos_dict,
    filter_processed_data,
    get_min_frames_count,
    generate_balanced_kfold_yolo,
    generate_random_kfold_yolo,
    get_experiment_splits_df
)
PROJECT_ROOT = Path().resolve().parent
CONFIG_PATH = PROJECT_ROOT / "paths.yaml"

In [2]:
ds_info = scan_raw_dataset(CONFIG_PATH)

# Transformación de los datos

El dataset original (`dataset/raw`) contiene videos `.mp4` y anotaciones en texto plano. 
El formato de estas etiquetas originales es:

`[frame_id, target_id, top_left_x, top_left_y, width, confidence, class, visibility]`

El tracking de YOLO requiere un formato muy específico:
- Una carpeta para imágenes (`processed/images`) y otra para etiquetas (`processed/labels`).
- Un archivo `.txt` por cada imagen (frame).
- Las coordenadas deben estar normalizadas entre [0, 1] y seguir el formato:

`[class_id, target_id, x_center, y_center, width, height]`

Para solucionar esto, aplicaremos la función `process_video_to_yolo`, la cual iterará sobre cada vídeo, extraerá sus frames y calculará las coordenadas relativas al tamaño del frame.

In [3]:
example = ds_info['data']['horse'][0]
video_path = example['video']
label_path = example['label']

gt_raw = load_gt(label_path)
gt_data = parse_annotations(gt_raw)

In [4]:
classes_dict = ds_info['classes'] # Viene del paths.yaml
id_map = {}
for yolo_index, (name, original_id) in enumerate(classes_dict.items()):
    id_map[original_id] = yolo_index
    
print(f"Mapa de IDs aplicado: {id_map}")

Mapa de IDs aplicado: {6: 0, 7: 1, 8: 2}


Se ha  creado el dataset con tracking

In [9]:
import gc

print(f"Ruta de destino: {ds_info['processed_path']}")


for class_name in ds_info['classes'].keys():
# for class_name in ['horse', 'pig']:
    print(f"\n> Procesando clase: {class_name.upper()}")
    
    lista_videos = ds_info['data'][class_name]
    
    for i, item in enumerate(lista_videos):
        video_path = item['video']
        label_path = item['label']
        
        base_name = Path(video_path).stem

        output_images = ds_info['processed_path'] / f'images' / base_name
        output_labels = ds_info['processed_path'] / f'labels_track' / base_name
        
        process_video_to_yolo(
            video_path=video_path, 
            gt_path=label_path, 
            output_img_dir=output_images, 
            output_lbl_dir=output_labels,
            class_map=id_map,
            label_tracks_bool=True,
            frame_step=5
        )
        gc.collect() 

Ruta de destino: /home/cgonzalez/Object_Recognition/dataset/processed

> Procesando clase: HORSE
Successfully processed horse_1.mp4: 56 frames saved.
Successfully processed horse_2.mp4: 62 frames saved.
Successfully processed horse_3.mp4: 286 frames saved.
Successfully processed horse_4.mp4: 41 frames saved.
Successfully processed horse_5.mp4: 65 frames saved.
Successfully processed horse_6.mp4: 63 frames saved.
Successfully processed horse_7.mp4: 263 frames saved.

> Procesando clase: PENGUIN
Successfully processed penguin_1.mp4: 62 frames saved.
Successfully processed penguin_2.mp4: 65 frames saved.
Successfully processed penguin_3.mp4: 64 frames saved.
Successfully processed penguin_4.mp4: 65 frames saved.
Successfully processed penguin_5.mp4: 64 frames saved.
Successfully processed penguin_6.mp4: 46 frames saved.

> Procesando clase: PIG
Successfully processed pig_1.mp4: 63 frames saved.
Successfully processed pig_2.mp4: 62 frames saved.
Successfully processed pig_3.mp4: 62 frames 

Ahora por agilizar vamos a realizar en la siguiente celda la creación de los experimentos como se hizo en el notebook `03_splid_ds_yolo.ipynb` pero solo para el escenario 3, aunque luego nos centremos solo en uno de los folds.

In [7]:
processed_path = ds_info['processed_path']
class_names = list(ds_info['classes'].keys())
processed_pool = get_processed_videos_dict(processed_path, class_names, images_folder="images_track_2_frames")

anomalies = ["penguin_3"]
filtered_dict, _ = filter_processed_data(processed_pool,  anomalies)

min_frames_dataset = get_min_frames_count(filtered_dict)
print(f"Vídeos tras filtrado: {sum(len(v) for v in filtered_dict.values())}")
print(f"Mínimo de frames detectado para balanceo: {min_frames_dataset}")

experiments_dir = processed_path / "experiments_track_2_frames"
SEED = 42

generate_random_kfold_yolo(
    data_dict=filtered_dict,
    output_path=experiments_dir,
    set_name="set3_random_full",
    k_folds=5,
    class_names=class_names,
    subsample=False,
    seed = SEED
)

Vídeos tras filtrado: 17
Mínimo de frames detectado para balanceo: 103
✅ Random/Unbalanced YOLO configuration files generated successfully in: /home/cgonzalez/Object_Recognition/dataset/processed/experiments_track_2/set3_random_full


In [8]:
processed_path = ds_info['processed_path']
class_names = list(ds_info['classes'].keys())
processed_pool = get_processed_videos_dict(processed_path, class_names, images_folder="images_track_5_frames")

anomalies = ["penguin_3"]
filtered_dict, _ = filter_processed_data(processed_pool,  anomalies)

min_frames_dataset = get_min_frames_count(filtered_dict)
print(f"Vídeos tras filtrado: {sum(len(v) for v in filtered_dict.values())}")
print(f"Mínimo de frames detectado para balanceo: {min_frames_dataset}")

experiments_dir = processed_path / "experiments_track_5_frames"
SEED = 42

generate_random_kfold_yolo(
    data_dict=filtered_dict,
    output_path=experiments_dir,
    set_name="set3_random_full",
    k_folds=5,
    class_names=class_names,
    subsample=False,
    seed = SEED
)

Vídeos tras filtrado: 17
Mínimo de frames detectado para balanceo: 41
✅ Random/Unbalanced YOLO configuration files generated successfully in: /home/cgonzalez/Object_Recognition/dataset/processed/experiments_track_5_frames/set3_random_full
